# rlmflow coding-agent walkthrough

Build a recursive coding agent, run a task, save the trace, and embed the generated artifact inline. This mirrors `examples/coding-agent/agent.py` but as a notebook so you can poke at every step.

Sibling notebooks read offline against the deterministic fixture under `examples/data/notebook-coding-agent/` (regenerated by `examples/data/build_notebook_fixture.py`):

- [`node_basics.ipynb`](./node_basics.ipynb) — querying the trace: `graph.walk`, `graph.find`, `graph.where`, `session.load_graph`, …
- [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb) — visualizing the trace: `tree`, `gantt`, `code_log`, mermaid / dot / d2, `report_md`, the inline Gradio viewer, etc.

Running this notebook end-to-end requires `OPENAI_API_KEY` and a live LLM call. Skip ahead to the other notebooks if you just want to consume the saved trace.

## 1. Build the agent

Four pieces snap together:

- `Workspace` — durable, branchable storage for sessions, traces, and any files the agent writes.
- `Runtime` — where REPL code runs. `LocalRuntime` runs in-process; `DockerRuntime` runs each step in a container.
- Tools — plain Python functions registered on the runtime. `FILE_TOOLS` gives the agent `read_file`, `write_file`, `edit_file`, `ls`, `grep`, etc.
- `RLMFlow` — wires an LLM client (plus optional cheaper alternates) to the runtime + workspace, and exposes `start` / `step` / `fork`. Every call to `start` / `step` returns a fresh immutable `Graph`.

In [1]:
from pathlib import Path

from rlmflow.llm import OpenAIClient
from rlmflow.rlm import RLMConfig, RLMFlow
from rlmflow.runtime.local import LocalRuntime
from rlmflow.tools import FILE_TOOLS
from rlmflow.workspace import Workspace

def build_agent(
    workspace_dir: str | Path = "./notebook-coding-agent",
    max_depth: int = 3,
    max_iterations: int = 30,
    max_concurrency: int = 8,
) -> RLMFlow:
    """Construct a coding agent identical to examples/coding-agent/agent.py."""
    workspace = Workspace.create(Path(workspace_dir).resolve())
    runtime = LocalRuntime(workspace=workspace)
    runtime.register_tools(FILE_TOOLS)

    return RLMFlow(
        llm_client=OpenAIClient("gpt-5-mini"),
        runtime=runtime,
        workspace=workspace,
        config=RLMConfig(
            max_depth=max_depth,
            max_iterations=max_iterations,
            max_concurrency=max_concurrency,
        ),
    )

## 2. Run a task

`agent.start(query)` returns the initial `Graph` (just the root agent with its `QueryNode`). Every `agent.step(graph)` advances exactly one tick — one LLM call, one REPL execution, or a wait-resolution — and returns a freshly-loaded `Graph` snapshot. Use `graph.current()` to inspect the latest state, `graph.finished` to stop, `graph.result()` for the terminal answer.

`live_view()` opens a self-updating Rich tree; you own the loop and just hand it the latest `Graph` on each tick. The agent loop and the visualization are decoupled.

In [2]:
TASK = """Create a boids simulation in plain html and javascript. Render hundreds of fast moving and colorful boids on a 2D canvas. Split each component into separate files. Verify each component after they are written. The main runnable interface should be in `index.html`. Make sure to save all components within ONLY this directory: `./output/boids-simulation/`.
"""

agent = build_agent(max_depth=2, workspace_dir="./boids-sim-workspace")
graph = agent.start(TASK)
graphs = [graph]

In [3]:
print(agent.build_system_prompt(graph))

## Role

You are a recursive agent with a Python REPL. You solve tasks by writing and executing Python programs, and you can delegate subtasks to sub-agents with their own fresh context windows.

**Why recurse?** Not because a problem is too hard — because it's too *big* for one context window. Each child you spawn via `delegate(...)` gets a fresh context budget. You get back its compact answer, not all the raw material.

**Reach for `delegate(...)`** when the user asks for **multiple files / components / pages**, when a long input wants **parallel chunked analysis**, or when two analyses are **independent** and want fresh context windows. Inline only when the artifact is small and tightly coupled.

## REPL

- Every response is exactly one ```repl``` code block. Tools are pre-imported; variables persist across turns within one agent.
- Final answer: `done(answer)` exactly once when complete. That string is what the parent/user sees.
- `yield wait(*handles)` ends the block. The runtime 

In [4]:
from rlmflow.utils.viz import live_view

with live_view() as view:
    view(graph)
    while not graph.finished:
        graph = agent.step(graph)
        graphs.append(graph)
        view(graph)

Output()

In [5]:
current = graph.current()
print(f"{len(graphs)} snapshots  \u00b7  final: {graph.root_agent_id} [{current.type}]")
print(f"query : {graph.query[:120]!r}...")
print(f"result: {graph.result()[:200]}")

8 snapshots  ·  final: root [done_output]
query : 'Create a boids simulation in plain html and javascript. Render hundreds of fast moving and colorful boids on a 2D canvas'...
result: Wrote boids simulation to ./output/boids-simulation/ — files: index.html, style.css, src/utils.js, src/boid.js, src/main.js. Open ./output/boids-simulation/index.html in a browser to run. If file:// c


In [6]:
print(graph.tree())

● root (default) — Create a boids simulation in plain html and javascript. Ren…
  - [ 0] query
  - [ 1] llm
  - [ 2] llm code=# Create a multi-file boids simulation.…
  - [ 3] exec
  - [ 4] errored (exec_exception)
  - [ 5] llm
  - [ 6] llm code=# We'll create the boids simulation fil…
  - [ 7] exec
  - [ 8] supervising waiting_on=['root.write_index', 'root.write_style', 'root.write_utils', 'root.write_boid', 'root.write_main']
    ● root.write_index (default) — Write file './output/boids-simulation/index.html' with the …
      - [ 0] query
      - [ 1] llm
      - [ 2] llm code=content = CONTEXT.read() write_file("./…
      - [ 3] exec
      - [ 4] done -> Wrote ./output/boids-simulation/index.html
    ● root.write_style (default) — Write file './output/boids-simulation/style.css' with the p…
      - [ 0] query
      - [ 1] llm
      - [ 2] llm code=# Read the provided context and write i…
      - [ 3] exec
      - [ 4] done -> Wrote ./output/boids-simulation/style.css
    ● root.writ

In [7]:
# from rlmflow.utils.trace import save_trace

# trace_path = save_trace(graphs, agent.workspace.trace_dir, metadata={"task": TASK})
# print(f"trace -> {trace_path}  ({len(graphs)} snapshots)")

## 3. Embed the generated artifact

Inline the `index.html` the agent produced (resolving CSS / ES-module imports / plain `<script src=>` tags) and render it via `<iframe srcdoc=...>` so it works inside Jupyter (which does not resolve relative iframe URLs).

In [8]:
import html as _html
import re
from IPython.display import HTML

live_candidates = sorted(agent.workspace.root.glob("output/**/index.html"))
canonical_candidates = sorted(
    Path("../data/boids-sim-workspace").glob("**/index.html")
)
candidates = live_candidates if live_candidates else canonical_candidates
if not candidates:
    raise FileNotFoundError(f"no index.html under {agent.workspace.root}")
html_path = candidates[-1]
base = html_path.parent


def _inline_css(m):
    p = base / m.group(1)
    return f"<style>\n{p.read_text()}\n</style>" if p.exists() else m.group(0)


def _inline_plain_js(m):
    p = base / m.group(1)
    return f"<script>\n{p.read_text()}\n</script>" if p.exists() else m.group(0)


def _flatten_module(entry: str) -> str:
    visited: set = set()
    chunks: list[str] = []

    def visit(path):
        path = path.resolve()
        if path in visited or not path.exists():
            return
        visited.add(path)
        src = path.read_text()
        for m in re.finditer(
            r"^\s*import\b.*?\bfrom\s+['\"]([^'\"]+)['\"]",
            src,
            flags=re.MULTILINE,
        ):
            visit(path.parent / m.group(1))
        src = re.sub(
            r"^\s*import\b.*?\bfrom\s+['\"][^'\"]+['\"];?\s*$",
            "",
            src,
            flags=re.MULTILINE,
        )
        src = re.sub(r"^\s*export\s+default\s+", "", src, flags=re.MULTILINE)
        src = re.sub(r"^\s*export\s+", "", src, flags=re.MULTILINE)
        chunks.append(f"// === {path.name} ===\n{src}")

    visit(base / entry)
    return "<script>\n" + "\n".join(chunks) + "\n</script>"


content = html_path.read_text()
content = re.sub(
    r'<link\b[^>]*\bhref="([^"]+)"[^>]*/?>', _inline_css, content, flags=re.I
)
content = re.sub(
    r'<script\b(?=[^>]*\btype="module")[^>]*\bsrc="([^"]+)"[^>]*>\s*</script>',
    lambda m: _flatten_module(m.group(1)),
    content,
    flags=re.I,
)
content = re.sub(
    r'<script\b[^>]*\bsrc="([^"]+)"[^>]*>\s*</script>',
    _inline_plain_js,
    content,
    flags=re.I,
)

print(f"rendering {html_path}")
HTML(
    f'<iframe srcdoc="{_html.escape(content, quote=True)}" '
    f'width="100%" height="600" '
    f'style="border:1px solid #ddd; border-radius:6px; background:#0a0a0f"></iframe>'
)

rendering /Users/shyam/Code/rlmkit/examples/notebooks/boids-sim-workspace/output/boids-simulation/index.html


/Users/shyam/anaconda3/envs/py312/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## 4. Open the interactive viewer

`open_viewer(graphs)` boots a Gradio stepper (step slider + clickable graph + per-agent transcript) inline.

In [1]:
from rlmflow.utils.viewer import open_viewer

open_viewer("./boids-sim-workspace", inline=True, height=720, quiet=True)

In [2]:
from rlmflow.utils.viewer import save_steps

save_steps(
    "./boids-sim-workspace",
    "./boids-sim-workspace/frames",
    width=1600,
    height=1200,
    scale=2,
    marker_mult=3.5,
    text_mult=2.2,
)

PosixPath('boids-sim-workspace/frames')

## Next

- [`node_basics.ipynb`](./node_basics.ipynb) — walk the saved trace with the `Graph` query API.
- [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb) — inline Plotly, Mermaid, DOT, Gantt, report exports.